## Imports

In [1]:
import os
import pandas as pd
import numpy as np
import urllib.request
import zipfile
import shutil
try:
    import boto3
except:
    ! pip install boto3
    import boto3
try:
    import pyarrow
except:
    ! pip install pyarrow
    import pyarrow

## Constants

In [2]:
str_bucket = 'recommender-system-demo'
str_task = '00_data_collection'
str_dirname_output = './output'
str_url = 'https://files.grouplens.org/datasets/movielens/ml-1m.zip'
str_zip_filename = 'ml-1m.zip'
str_extract_dir = 'ml-1m'

## Output Directory

In [3]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

## Download MovieLens 1M

In [4]:
# download zip
print(f'Downloading {str_url}...')
urllib.request.urlretrieve(str_url, str_zip_filename)
print('Download complete.')

# extract
print('Extracting...')
with zipfile.ZipFile(str_zip_filename, 'r') as zip_ref:
    zip_ref.extractall('.')
print('Extraction complete.')

Download complete.
Extracting...
Extraction complete.


## Parse Data Files

In [5]:
# ratings
print('Parsing ratings.dat...')
df_ratings = pd.read_csv(
    f'{str_extract_dir}/ratings.dat',
    sep='::',
    header=None,
    names=['user_id', 'movie_id', 'rating', 'timestamp'],
    engine='python',
)
df_ratings['timestamp'] = pd.to_datetime(df_ratings['timestamp'], unit='s')
print(f'  Ratings: {df_ratings.shape[0]:,} rows, {df_ratings.shape[1]} columns')

# users
print('Parsing users.dat...')
df_users = pd.read_csv(
    f'{str_extract_dir}/users.dat',
    sep='::',
    header=None,
    names=['user_id', 'gender', 'age', 'occupation', 'zip_code'],
    engine='python',
)
print(f'  Users: {df_users.shape[0]:,} rows, {df_users.shape[1]} columns')

# movies
print('Parsing movies.dat...')
df_movies = pd.read_csv(
    f'{str_extract_dir}/movies.dat',
    sep='::',
    header=None,
    names=['movie_id', 'title', 'genres'],
    engine='python',
    encoding='latin-1',
)
print(f'  Movies: {df_movies.shape[0]:,} rows, {df_movies.shape[1]} columns')

Parsing ratings.dat...
  Ratings: 1,000,209 rows, 4 columns
Parsing users.dat...
  Users: 6,040 rows, 5 columns
Parsing movies.dat...
  Movies: 3,883 rows, 3 columns


## Data Summary

In [6]:
int_n_ratings = df_ratings.shape[0]
int_n_users = df_ratings['user_id'].nunique()
int_n_movies = df_ratings['movie_id'].nunique()
flt_sparsity = 1 - int_n_ratings / (int_n_users * int_n_movies)

print(f'Total ratings:  {int_n_ratings:,}')
print(f'Unique users:   {int_n_users:,}')
print(f'Unique movies:  {int_n_movies:,}')
print(f'Rating range:   {df_ratings["rating"].min()} - {df_ratings["rating"].max()}')
print(f'Mean rating:    {df_ratings["rating"].mean():.2f}')
print(f'Date range:     {df_ratings["timestamp"].min()} to {df_ratings["timestamp"].max()}')
print(f'Sparsity:       {flt_sparsity:.4%}')

Total ratings:  1,000,209
Unique users:   6,040
Unique movies:  3,706
Rating range:   1 - 5
Mean rating:    3.58
Date range:     2000-04-25 23:05:32 to 2003-02-28 17:49:50
Sparsity:       95.5316%


## Save to S3 as Parquet

In [7]:
str_uri_ratings = f's3://{str_bucket}/{str_task}/ratings.parquet'
str_uri_users = f's3://{str_bucket}/{str_task}/users.parquet'
str_uri_movies = f's3://{str_bucket}/{str_task}/movies.parquet'

print(f'Saving ratings to {str_uri_ratings}...')
df_ratings.to_parquet(str_uri_ratings, index=False)

print(f'Saving users to {str_uri_users}...')
df_users.to_parquet(str_uri_users, index=False)

print(f'Saving movies to {str_uri_movies}...')
df_movies.to_parquet(str_uri_movies, index=False)

print('All files saved to S3.')

Saving ratings to s3://recommender-system-demo/00_data_collection/ratings.parquet...
Saving users to s3://recommender-system-demo/00_data_collection/users.parquet...
Saving movies to s3://recommender-system-demo/00_data_collection/movies.parquet...
All files saved to S3.


## Cleanup

In [8]:
# remove downloaded zip and extracted directory
os.remove(str_zip_filename)
shutil.rmtree(str_extract_dir)
print('Local files cleaned up.')

Local files cleaned up.
